### Silver Layer - What Is It?

In our project, data moves through 3 layers: **Bronze → Silver → Gold**

**Bronze** = Raw data dumped straight from files. No cleaning, no changes. Just a copy of what we got.

**Silver** = Cleaned, fixed, and organized version of that same data. This is where we do the real work.

**Gold** = Final reports and dashboards built on top of Silver.

---

### What exactly happens in Silver?

We take each Bronze table and fix all the problems:

| Problem in Bronze | What we do in Silver |
| --- | --- |
| Extra columns we don't need (like `url`) | Drop them |
| Column names are messy (`circuitId`, `raceName`) | Rename to snake_case (`circuit_id`, `race_name`) |
| Some rows have missing key values | Remove them (null filter) |
| Same row appears twice | Remove duplicates |
| Text is inconsistent ("british" vs "British") | Apply title case to fix it |
| No timestamp tracking | Add `created_timestamp` and `updated_timestamp` |

---

### How does the data get saved?

We use a helper function called `write_to_silver()` which handles two situations:

1. **First time** (table doesn't exist) → creates the table and writes everything
2. **Next time** (table already exists) → uses MERGE to:
   - **Update** rows that changed (if new batch_id is newer)
   - **Insert** rows that are brand new
   - **Keep** old rows that didn't change

This means we never lose data and always have the latest version.

---

### Utilities used (from `00-common` folder):

| File | What it does |
| --- | --- |
| `01.environment-config` | Stores shared variables: `catalog_name` = `formula1_incr`, `bronze_schema` = `bronze`, `silver_schema` = `silver` |
| `03.silver_helpers` | Contains the `write_to_silver()` function that all silver notebooks call to save data |

Every silver notebook runs these two files first using `%run` so the variables and function are available.

---
### Entity Relationship Diagram - Silver Layer (`formula1_incr.silver`)

After cleaning, we have 6 tables in the silver schema. Here's how they connect:

**Main tables:**
- `circuits` — all F1 tracks (77 circuits)
- `races` — every race that happened (season, round, date, which circuit)
- `constructors` — teams like Mercedes, Ferrari, Red Bull
- `drivers` — all drivers with their name, nationality, date of birth
- `results` — race results (who finished where, how many points)
- `sprints` — sprint race results (same structure as results)

**How they connect:**

| From | To | How | Meaning |
| --- | --- | --- | --- |
| `races` | `circuits` | `circuit_id` | Each race happens at one circuit |
| `results` | `drivers` | `driver_id` | Each result belongs to one driver |
| `results` | `constructors` | `constructor_id` | Each result belongs to one team |
| `sprints` | `drivers` | `driver_id` | Each sprint result belongs to one driver |
| `sprints` | `constructors` | `constructor_id` | Each sprint result belongs to one team |
| `results` / `sprints` | `races` | `season` + `round` | Links results to the specific race |

**Note:** `results` and `sprints` don't have a direct `circuit_id` — they connect to circuits through the `races` table.

**Figma Source:** [Formula1 DataWarehouse ERD](https://www.figma.com/design/qwT3ax5c5f804QC3Dqjnbt/Formula1_DataWarehouse_ERD?node-id=0-1&p=f&m=draw)

---
### Code Flow - How Every Silver Notebook Works

All silver notebooks follow the same pattern. Here's each step explained:

##### Step 1: Load Config & Helpers
```python
%run ../00-common/01.environment-config
%run ../00-common/03.silver_helpers
```
- Loads `catalog_name`, `bronze_schema`, `silver_schema` variables
- Loads the `write_to_silver()` function
- Without these, nothing else works — they must run first

##### Step 2: Set Table Names
```python
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'
```
- Builds the full table name like `formula1_incr.bronze.circuits`
- `bronze_table` = where we read FROM
- `silver_table` = where we write TO

##### Step 3: Import Functions
```python
from pyspark.sql.functions import *
```
- Imports all Spark functions we need: `col()`, `initcap()`, `concat_ws()`, `current_timestamp()`, etc.
- The `*` means "import everything" so we don't have to list each one

##### Step 4: Read Bronze Table
```python
df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)
```
- Reads all data from the bronze table into a DataFrame
- Filters to only the current batch (so we process one batch at a time, not everything)
- `v_batch_id` comes from a widget parameter passed when the notebook runs

##### Step 5: Drop Unwanted Columns
```python
df_drop = df.drop('url')
```
- Removes columns that have no value for analysis
- Example: `url` is just a Wikipedia link — useless for data analysis
- You can also use `.select()` to pick only the columns you want (opposite approach)

##### Step 6: Rename Columns
```python
# Rename one column
df_renamed = df.withColumnRenamed('old_name', 'new_name')

# Rename many columns at once (better)
df_renamed = df.withColumnsRenamed({
    'circuitId': 'circuit_id',
    'raceName': 'race_name'
})
```
- Changes column names from camelCase to snake_case
- Why? Because snake_case is the standard in data engineering — easier to read and consistent
- `withColumnsRenamed()` (with S) is better when renaming multiple columns

##### Step 7: Remove Rows with Null Keys
```python
df_valid = df.filter(col('circuit_id').isNotNull())

# Multiple conditions
df_valid = df.filter(
    col('season').isNotNull() &
    col('driver_id').isNotNull()
)
```
- If a row is missing its key column (like `circuit_id`), it's useless — we can't join it to anything
- We remove these bad rows to keep data quality high
- Use `&` for AND conditions, `|` for OR conditions

##### Step 8: Remove Duplicates
```python
# Remove rows that are 100% identical
df_distinct = df.distinct()

# Remove rows with same key (keeps first occurrence)
df_distinct = df.dropDuplicates(['circuit_id'])
```
- Sometimes the same data gets loaded twice (duplicate batches, overlapping files)
- `distinct()` removes rows where ALL columns match
- `dropDuplicates(['key'])` removes rows where just the key matches — more targeted

##### Step 9: Apply Title Case
```python
df_final = df.withColumn('name', initcap(col('name')))
```
- `initcap()` capitalizes the first letter of each word
- "british" → "British", "red bull racing" → "Red Bull Racing"
- Makes text consistent and clean for reports/dashboards

##### Step 10: Concatenate Columns (Drivers only)
```python
df = df.withColumn('driver_name',
    initcap(concat_ws(' ', col('name.givenName'), col('name.familyName'))))
```
- Only used in the Drivers notebook because names come as nested JSON: `{givenName: "lewis", familyName: "hamilton"}`
- `concat_ws(' ', ...)` joins values with a space → "lewis hamilton"
- Then `initcap()` makes it → "Lewis Hamilton"
- After this, we drop the original nested `name` column since we don't need it anymore

##### Step 11: Write to Silver Table
```python
write_to_silver(
    input_df = df_final,
    target_table = silver_table,
    merge_condition = 't.circuit_id = s.circuit_id',
    columns_to_update = ['circuit_name', 'latitude', 'longitude', ...]
)
```
- Calls our helper function from `03.silver_helpers`
- `input_df` = the cleaned DataFrame we prepared
- `target_table` = where to save it (e.g., `formula1_incr.silver.circuits`)
- `merge_condition` = how to match old rows with new rows (usually by the primary key)
- `columns_to_update` = which columns to refresh when a row already exists

Internally, it:
1. Adds `created_timestamp` and `updated_timestamp` columns
2. Checks if the table exists
3. If NO → creates the table from scratch
4. If YES → merges new data in (update existing + insert new)

---

### Summary of All Silver Notebooks:

| Notebook | Source Table | Silver Table | Special Steps |
| --- | --- | --- | --- |
| `01) Circuits` | `bronze.circuits` | `silver.circuits` | Drop url, rename lat/long, title case locality |
| `02) Races` | `bronze.races` | `silver.races` | Drop url, rename raceName, title case |
| `03) Constructors` | `bronze.constructors` | `silver.constructors` | Drop url, rename constructorId, title case nationality |
| `04) Drivers` | `bronze.drivers` | `silver.drivers` | Concatenate nested name fields, title case |
| `05) Results` | `bronze.results` | `silver.results` | Rename 9 columns, filter 4 null keys, title case race_name |
| `06) Sprints` | `bronze.sprints` | `silver.sprints` | Same as Results but for sprint races |